Create a table of original and revival blush products.
*Co-authored with CoCo*

In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

for weight_file in ['ClashGrotesk-Extralight.ttf', 'ClashGrotesk-Light.ttf',
                    'ClashGrotesk-Regular.ttf', 'ClashGrotesk-Medium.ttf',
                    'ClashGrotesk-Semibold.ttf', 'ClashGrotesk-Bold.ttf']:
    fm.fontManager.addfont(f'fonts/Clash_Grotesk/{weight_file}')

plt.rcParams['font.family'] = 'Clash Grotesk'
plt.rcParams['font.weight'] = 'light'

In [ ]:
%%sql -r dataframe_1
USE DATABASE MARCJACOBS_BEAUTY_REVIVAL;

In [ ]:
%%sql -r dataframe_2
SELECT
  s.era,
  s.launch_year,
  s.product_name,
  s.product_type,
  s.shade_code,
  s.shade_name,
  s.shade_description,
  s.finish,
  s.finish_group,
  s.color_family,
  s.shade_depth_group,
  s.undertone_inferred_cmyk,
  s.undertone_confidence,
  s.c_pct,
  s.m_pct,
  s.y_pct,
  s.k_pct,
  s.color_value_source
FROM MARCJACOBS_BEAUTY_REVIVAL.ANALYTICS.SHADES s
LEFT JOIN MARCJACOBS_BEAUTY_REVIVAL.ANALYTICS.PRODUCTS p
  ON s.era = p.era
 AND s.product_name = p.product_name
WHERE s.product_type ILIKE '%Blush%'
  AND s.shade_name IS NOT NULL
ORDER BY s.era, s.product_name, s.shade_name;

In [ ]:
import pandas as pd
from matplotlib.lines import Line2D

def cmyk_to_rgb(c, m, y, k):
    c, m, y, k = c / 100, m / 100, y / 100, k / 100
    r = (1 - c) * (1 - k)
    g = (1 - m) * (1 - k)
    b = (1 - y) * (1 - k)
    return (r, g, b)

df = dataframe_2.to_pandas() if hasattr(dataframe_2, 'to_pandas') else dataframe_2
df = df.copy()
df.columns = [col.lower() for col in df.columns]

df['rgb'] = df.apply(lambda row: cmyk_to_rgb(row['c_pct'], row['m_pct'], row['y_pct'], row['k_pct']), axis=1)

def map_finish(finish):
    if finish == 'Natural':
        return 'Cream'
    elif finish == 'Matte':
        return 'Powder'
    return 'other'

df['finish_label'] = df['finish'].apply(map_finish)

shape_map = {
    'Cream': 'o',
    'Powder': 's',
    'other': '^',
}

highlight_shades = {'Attention, Please', 'Cherry Tease', 'Pink Kink', 'Hot Flush', 'Last Word'}
era_order = ['original_launch', 'revival']
era_labels = {'original_launch': '2013 Collection', 'revival': '2026 Revival'}
era_annotations = {
    'original_launch': 'Powder blushes cluster in muted rose and berry tones',
    'revival': 'Cream blush expands into hot pink, orange, coral, and plum',
}

def plot_blush(layout='horizontal'):
    if layout == 'horizontal':
        fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex=True, sharey=True)
        order = era_order
    else:
        fig, axes = plt.subplots(2, 1, figsize=(9, 14), sharex=True)
        order = era_order

    for i, (axis, era) in enumerate(zip(axes.flat, order)):
        era_df = df[df['era'] == era].copy()

        for finish_label, finish_df in era_df.groupby('finish_label'):
            colors = list(finish_df['rgb'])
            marker = shape_map.get(finish_label, '^')
            axis.scatter(
                finish_df['m_pct'],
                finish_df['y_pct'],
                s=140,
                alpha=0.9,
                c=colors,
                edgecolors='black',
                linewidths=0.6,
                marker=marker,
                label=finish_label,
            )

        for _, row in era_df.iterrows():
            if row['shade_name'] in highlight_shades:
                axis.annotate(
                    row['shade_name'],
                    (row['m_pct'], row['y_pct']),
                    textcoords='offset points',
                    xytext=(6, 5),
                    fontsize=9,
                    fontweight='normal',
                    alpha=0.9,
                )

        axis.set_title(era_labels[era], fontsize=13, fontweight='medium', pad=20)
        axis.text(0.5, 1.01, era_annotations[era],
                  transform=axis.transAxes, ha='center', fontsize=11,
                  fontweight='normal', color='#444')
        axis.set_xlabel('Pink / magenta intensity', fontsize=11, fontweight='normal')
        axis.grid(alpha=0.2)
        axis.tick_params(labelsize=9)

        if layout == 'vertical' or i == 0:
            axis.set_ylabel('Orange / coral intensity', fontsize=11, fontweight='normal')

    legend_handles = [
        Line2D([0], [0], marker='s', color='black', markerfacecolor='white',
               linestyle='', markersize=9, label='Powder'),
        Line2D([0], [0], marker='o', color='black', markerfacecolor='white',
               linestyle='', markersize=9, label='Cream'),
    ]
    if layout == 'horizontal':
        axes.flat[0].legend(handles=legend_handles, title='Finish / formula',
                            loc='lower left', bbox_to_anchor=(0, 1.15),
                            fontsize=10, title_fontsize=10,
                            frameon=False, borderaxespad=0)
        fig.suptitle('BLUSH CAME BACK BRIGHTER, CREAMIER, AND WARMER',
                     fontsize=18, fontweight='medium', y=0.95)
        plt.tight_layout()
    else:
        axes.flat[0].legend(handles=legend_handles, title='Finish / formula',
                            loc='lower left', bbox_to_anchor=(0, 1.08),
                            fontsize=10, title_fontsize=10,
                            frameon=False, borderaxespad=0)
        plt.tight_layout()
        plt.subplots_adjust(hspace=0.22, top=0.94)
        fig.suptitle('BLUSH CAME BACK BRIGHTER, CREAMIER, AND WARMER',
                     fontsize=18, fontweight='medium', y=1.05)

    plt.show()

plot_blush('horizontal')
plot_blush('vertical')